In [1]:
from datasets import load_dataset

ds = load_dataset("SWE-bench/SWE-smith-trajectories", split="xml")

In [2]:
df = ds.to_polars()
len(df)

26076

In [3]:
SEED = 42
MAX_EXAMPLES_PER_INSTANCE = 3

df2 = (
    df
    .sample(fraction=1.0, shuffle=True, seed=SEED)
    .group_by("instance_id")
    .head(MAX_EXAMPLES_PER_INSTANCE)
)
len(df2)

20585

In [4]:
import polars as pl

LOW_QUALITY_INSTANCE_TYPES = [
    ".lm_modify",
    ".combine",
    ".func",
]

df3 = df2.filter(
    ~pl.col("instance_id").str.contains(instance_type)
    for instance_type in LOW_QUALITY_INSTANCE_TYPES
)
len(df3)

10700

In [ ]:
TOKENS_PER_CHAR = 0.28
MODEL_MAX_LENGTH = 32768

df4 = df3.filter(pl.col("messages").str.len_chars() * TOKENS_PER_CHAR < MODEL_MAX_LENGTH / 2)
len(df4)

2860

In [ ]:
from pathlib import Path

path = Path("data", "swe_smith.parquet")
path.parent.mkdir(parents=True, exist_ok=True)

df4.write_parquet(path)